Load Virginia's per `.tfrecord` files and put them into a single `.h5` file. The scale cuts are applied here too.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pickle, h5py, os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# utils
from msi.utils import input_output, plotting, mcmc
from msfm.utils import prior, parameters, files

23-07-20 06:15:04    scales.py INF   Setting up healpy to run on 256 CPUs 


In [3]:
conf = files.load_config()

with h5py.File("/global/homes/a/athomsen/multiprobe-simulation-forward-model/data/CosmoGridV1_metainfo.h5", "r") as f:
    cosmogrid_info = f["parameters/grid"][:]
info_Om = cosmogrid_info["Om"]

out_dir = "/global/cfs/cdirs/des/athomsen/power_spectra/v3"

# clustering only

In [4]:
params = ["Om", "s8", "bg", "n_bg"]
out_file = os.path.join(out_dir, "cls_clustering.h5")

grid_dir = "/global/cfs/cdirs/des/vajani/combined-probes/output/Galaxy_Clustering/grid/noisy/Power_spectra"
fidu_dir = "/global/cfs/cdirs/des/vajani/combined-probes/output/Galaxy_Clustering/fiducial/noisy/Power_spectra"

grid_pattern = "GC_index_*_DESy3_grid_*.tfrecord_*.h5"
fidu_pattern = "GC_index_*_DESy3_fiducial_*.tfrecord.h5"

l_mins = conf["analysis"]["scale_cuts"]["clustering"]["l_min"]
l_maxs = conf["analysis"]["scale_cuts"]["clustering"]["l_max"]

n_z_bins = 4

In [5]:
# fiducial
fidu_index = []
fidu_cls = []

# grid
grid_theta = []
grid_cls = []

# loop over auto and cross bins
for i in range(n_z_bins):
    for j in range(n_z_bins):
        if i <= j:
            bin_num = f"{i+1}x{j+1}"
            print(f"\nLoading bin_num = {bin_num}")

            # load ells
            centers = np.load(f'/global/cfs/cdirs/des/vajani/combined-probes/output/Galaxy_Clustering/binning/GC_CENTERS_mode=E_stat=CrossCLs_tomo={bin_num}.npy')
            
            # scale cuts, always use the lower bin
            l_min = l_mins[i]
            l_max = l_maxs[i]
            l_cut = (l_min < centers) & (centers < l_max)
            n_summary = np.count_nonzero(l_cut)
            print(f"Using scale cuts l_min = {l_min}, l_max = {l_max}, which results in Cls of size {n_summary}")

            # fiducial #####################################################################################################################################
            fidu_file_list = glob.glob(fidu_dir + f"/bin{bin_num}/" + fidu_pattern)
            print(f"fiducial: found {len(fidu_file_list)} .h5 files")

            current_index = []
            current_cls = []

            for fidu_file in tqdm(fidu_file_list):
                with h5py.File(fidu_file, "r") as f:
                    current_index.append(f["power_spectrum"].attrs["index"][0,0])
                    current_cls.append(f["power_spectrum"][:])
                    
            current_index = np.asarray(current_index)
            current_cls = np.asarray(current_cls)
            
            # every example index must be unique
            assert len(np.unique(current_index)) == len(fidu_file_list)

            # sort
            i_sorted = np.argsort(current_index)
            current_index = current_index[i_sorted]
            current_cls = current_cls[i_sorted]
            
            # apply scale cut
            current_cls = current_cls[:,l_cut]
            
            # collect in lists
            fidu_index.append(current_index)
            fidu_cls.append(current_cls)

            # grid #####################################################################################################################################
            grid_file_list = glob.glob(grid_dir + f"/bin{bin_num}/" + grid_pattern)
            print(f"grid: found {len(grid_file_list)} .h5 files")

            current_theta = []
            current_cls = []

            for grid_file in tqdm(grid_file_list):
                with h5py.File(grid_file, "r") as f:
                    current_theta.append(np.squeeze(f["cosmo"][:]))
                    current_cls.append(f["power_spectrum"][:])
                    
            current_theta = np.asarray(current_theta)
            current_cls = np.asarray(current_cls)
            
            # sort (according to Om)
            i_sorted = np.argsort(current_theta[:,0])
            current_index = current_theta[i_sorted]
            current_cls = current_cls[i_sorted]

            # apply scale cut
            current_cls = current_cls[:,l_cut]
            
            # collect in lists
            grid_theta.append(current_theta)
            grid_cls.append(current_cls)


Loading bin_num = 1x1
Using scale cuts l_min = 30, l_max = 328, which results in Cls of size 14
fiducial: found 750 .h5 files


100%|██████████| 750/750 [00:01<00:00, 528.78it/s]


grid: found 39984 .h5 files


100%|██████████| 39984/39984 [02:16<00:00, 292.39it/s]



Loading bin_num = 1x2
Using scale cuts l_min = 30, l_max = 328, which results in Cls of size 14
fiducial: found 750 .h5 files


...

100%|██████████| 39984/39984 [01:38<00:00, 404.56it/s] 



Loading bin_num = 4x4
Using scale cuts l_min = 30, l_max = 739, which results in Cls of size 24
fiducial: found 750 .h5 files


100%|██████████| 750/750 [00:00<00:00, 1063.71it/s]


grid: found 39984 .h5 files


100%|██████████| 39984/39984 [03:43<00:00, 178.76it/s]


In [6]:
# fiducial #####################################################################################################################################

# collect the different examples
assert np.all([fidu_index[0] == fidu_index[i] for i in range(1, len(fidu_index))]), f"The fiducial indices should be identical for all bins"
fidu_index = fidu_index[0]

fidu_cls = np.concatenate(fidu_cls, axis=-1)

print(f"\nfidu_index.shape = {fidu_index.shape}")
print(f"fidu_cls.shape = {fidu_cls.shape}")

# grid #####################################################################################################################################

# collect the different cosmologies TODO uncomment
# assert np.all([grid_theta[0] == grid_theta[i] for i in range(1, len(grid_theta))]), f"The grid parameters should be identical for all bins"
grid_theta = grid_theta[0]

grid_cls = np.concatenate(grid_cls, axis=-1)

print(f"\ngrid_theta.shape = {grid_theta.shape}")
print(f"grid_cls.shape = {grid_cls.shape}")

# save the result in a single file
with h5py.File(out_file, "w") as f:
    f.create_dataset(name="fidu/index", data=fidu_index)
    f.create_dataset(name="fidu/cls", data=fidu_cls)
    
    f.create_dataset(name="grid/theta", data=grid_theta)
    f.create_dataset(name="grid/cls", data=grid_cls)


fidu_index.shape = (750,)
fidu_cls.shape = (750, 178)

grid_theta.shape = (39984, 10)
grid_cls.shape = (39984, 178)


# lensing only

In [11]:
params = ["Om", "s8", "Aia", "n_Aia"]
out_file = os.path.join(out_dir, "cls_lensing.h5")

grid_dir = "/global/cfs/cdirs/des/vajani/combined-probes/output/Weak_Lensing/grid/Power_Spectra/dictionaries"
fidu_dir = "/global/cfs/cdirs/des/vajani/combined-probes/output/Weak_Lensing/fiducial/noisy/Power_spectra"

fidu_pattern = "WL_index_*_DESy3_fiducial_*.tfrecord.h5"

l_mins = conf["analysis"]["scale_cuts"]["lensing"]["l_min"]
l_maxs = conf["analysis"]["scale_cuts"]["lensing"]["l_max"]

n_z_bins = 4

### fiducial

In [12]:
fidu_index = []
fidu_cls = []

for i in range(n_z_bins):
    for j in range(n_z_bins):
        if i <= j:
            bin_num = f"{i+1}x{j+1}"
            print(f"Loading bin_num = {bin_num}")

            # load ells
            centers = np.squeeze(np.load(f'/global/cfs/cdirs/des/vajani/Cori/DESY3/SUMMARY/CENTERS_mode=E_stat=CrossCLs_tomo={bin_num}.npy'))
                        
            # scale cuts
            l_min = l_mins[i]
            l_max = l_maxs[i]
            l_cut = (l_min < centers) & (centers < l_max)
            n_summary = np.count_nonzero(l_cut)
            print(f"Using scale cuts l_min = {l_min}, l_max = {l_max}, which results in Cls of size {n_summary}")

            # load cls from .h5
            fidu_file_list = glob.glob(fidu_dir + f"/bin{bin_num}/" + fidu_pattern)
            print(fidu_dir + f"/bin{bin_num}/" + fidu_pattern)
            print(f"found {len(fidu_file_list)} .h5 files")

            current_index = []
            current_cls = []

            for fidu_file in fidu_file_list:
                with h5py.File(fidu_file, "r") as f:
                    current_index.append(f["power_spectrum"].attrs["index"][0,0])
                    current_cls.append(f["power_spectrum"][:])
                    
            current_index = np.asarray(current_index)
            current_cls = np.asarray(current_cls)
            
            # every example index must be unique
            assert len(np.unique(current_index)) == len(fidu_file_list)

            # sort
            i_sorted = np.argsort(current_index)
            current_index = current_index[i_sorted]
            current_cls = current_cls[i_sorted]
            
            # apply scale cut
            current_cls = current_cls[:,l_cut]
            
            # collect in lists
            fidu_index.append(current_index)
            fidu_cls.append(current_cls)

# collect the different cosmologies
fidu_index = np.stack(fidu_index, axis=-1)
fidu_cls = np.concatenate(fidu_cls, axis=-1)

print(f"\nfidu_index.shape = {fidu_index.shape}")
print(f"fidu_cls.shape = {fidu_cls.shape}")

Loading bin_num = 1x1
Using scale cuts l_min = 30, l_max = 723, which results in Cls of size 16
/global/cfs/cdirs/des/vajani/combined-probes/output/Weak_Lensing/fiducial/noisy/Power_spectra/bin1x1/WL_index_*_DESy3_fiducial_*.tfrecord.h5
found 750 .h5 files
Loading bin_num = 1x2
Using scale cuts l_min = 30, l_max = 723, which results in Cls of size 16
/global/cfs/cdirs/des/vajani/combined-probes/output/Weak_Lensing/fiducial/noisy/Power_spectra/bin1x2/WL_index_*_DESy3_fiducial_*.tfrecord.h5
found 750 .h5 files
Loading bin_num = 1x3
Using scale cuts l_min = 30, l_max = 723, which results in Cls of size 16
/global/cfs/cdirs/des/vajani/combined-probes/output/Weak_Lensing/fiducial/noisy/Power_spectra/bin1x3/WL_index_*_DESy3_fiducial_*.tfrecord.h5
found 750 .h5 files
Loading bin_num = 1x4
Using scale cuts l_min = 30, l_max = 723, which results in Cls of size 16
/global/cfs/cdirs/des/vajani/combined-probes/output/Weak_Lensing/fiducial/noisy/Power_spectra/bin1x4/WL_index_*_DESy3_fiducial_*.tfre

### grid

In [13]:
from sobol_seq import i4_sobol

In [14]:
n_params = len(params)
n_examples = 16
n_summary = 32

grid_theta = []
grid_cls = []

n_summaries = []

for i in range(n_z_bins):
    for j in range(n_z_bins):
        if i <= j:
            bin_num = f"{i+1}x{j+1}"
            print(f"Loading bin_num = {bin_num}")
                        
            # load grid cls
            with open(os.path.join(grid_dir, f'bin{bin_num}/power_spec_dict_bin{bin_num}.pickle'), 'rb') as f:
                current_dict = pickle.load(f)
            
            # load ells
            centers = np.load(f'/global/cfs/cdirs/des/vajani/Cori/DESY3/SUMMARY/CENTERS_mode=E_stat=CrossCLs_tomo={bin_num}.npy').reshape(32)
            
            # scale cuts
            l_min = l_mins[i]
            l_max = l_maxs[i]
            l_cut = (l_min < centers) & (centers < l_max)
            n_summary = np.count_nonzero(l_cut)
            print(f"Using scale cuts l_min = {l_min}, l_max = {l_max}, which results in Cls of size {n_summary}")
            n_summaries.append(n_summary)
            
            # container
            current_cls = np.zeros((len(current_dict), n_examples, n_summary))

            # loop through the dictionary
            for k, (key, value) in tqdm(enumerate(current_dict.items()), total=len(current_dict.items())):
                # value is a list of length n_examples that contains arrays of shape (32,)
                Cls = np.asarray(value)[:,l_cut]
                # Cls = np.asarray(value)

                # only save the cosmological parameters once
                if i == 0 and j == 0:
                    current_theta_list = key.replace("[", "").replace("]", "").strip().split(" ")
                    current_theta_list = list(filter(None, current_theta_list))
                    current_theta_list = [float(item) for item in current_theta_list] 
                    
                    # build eta from the sobol sequence
                    current_Om = current_theta_list[0]
                    i_info = np.argwhere(np.isclose(current_Om, info_Om, rtol=1e-6, atol=1e-8))
                    assert i_info.shape == (1,1)
                    i_info = i_info[0,0]
                    i_sobol = cosmogrid_info["sobol_index"][i_info]
                    
                    sobol_point, _ = i4_sobol(8, i_sobol)
                    current_eta = sobol_point[7] * 10 - 4.0
                    current_theta_list.append(current_eta)                                      
                    
                    grid_theta.append(np.array(current_theta_list))
                                        
                current_cls[k] = np.stack(Cls, axis=0)
                
            grid_cls.append(current_cls)

# collect the different cosmologies
grid_theta = np.stack(grid_theta, axis=0)
grid_cls = np.concatenate(grid_cls, axis=-1)

print(f"\ngrid_theta.shape = {grid_theta.shape}")
print(f"grid_cls.shape = {grid_cls.shape}")

# remove the example axis
grid_cls = np.concatenate(grid_cls, axis=0)
grid_theta = np.repeat(grid_theta, grid_cls.shape[0]//grid_theta.shape[0], axis=0)

print(f"\ngrid_theta.shape = {grid_theta.shape}")
print(f"grid_cls.shape = {grid_cls.shape}")

# # collect the different cosmologies
# assert np.all([grid_theta[0] == grid_theta[i] for i in range(1, len(grid_theta))]), f"The grid parameters should be identical for all bins"
# grid_theta = grid_theta[0]

# grid_cls = np.concatenate(grid_cls, axis=-1)

# print(f"\ngrid_theta.shape = {grid_theta.shape}")
# print(f"grid_cls.shape = {grid_cls.shape}")

Loading bin_num = 1x1
Using scale cuts l_min = 30, l_max = 723, which results in Cls of size 16


100%|██████████| 2499/2499 [02:33<00:00, 16.31it/s] 


Loading bin_num = 1x2
Using scale cuts l_min = 30, l_max = 723, which results in Cls of size 16


100%|██████████| 2499/2499 [00:00<00:00, 50538.90it/s]


Loading bin_num = 1x3
Using scale cuts l_min = 30, l_max = 723, which results in Cls of size 16


...

Loading bin_num = 3x4
Using scale cuts l_min = 30, l_max = 1408, which results in Cls of size 24


100%|██████████| 2499/2499 [00:00<00:00, 50801.49it/s]


Loading bin_num = 4x4
Using scale cuts l_min = 30, l_max = 1535, which results in Cls of size 25


100%|██████████| 2499/2499 [00:00<00:00, 48427.12it/s]



grid_theta.shape = (2499, 4)
grid_cls.shape = (2499, 16, 197)

grid_theta.shape = (39984, 4)
grid_cls.shape = (39984, 197)


In [15]:
# save the result in a single file
with h5py.File(out_file, "w") as f:
    f.create_dataset(name="fidu/index", data=fidu_index)
    f.create_dataset(name="fidu/cls", data=fidu_cls)
    
    f.create_dataset(name="grid/theta", data=grid_theta)
    f.create_dataset(name="grid/cls", data=grid_cls)